# 🛍️ Lumière EU E-Commerce — End-to-End Analytics Pipeline
### AIDVS Executive Master · Porto Business School · Group Project

**Business question:** *"Where should Lumière focus its commercial attention over the next two quarters to grow profitably?"*

| Detail | Value |
|---|---|
| Dataset | 82 000 orders · 5 200 customers · 177 products · 6 150 returns |
| Period | Jan 2024 – Dec 2025 |
| Markets | 8 EU countries · 3 channels |
| Currency | EUR (€) |

---

## Notebook Structure

| Phase | Section | Description |
|---|---|---|
| 0 | Setup | Imports, config, constants |
| 1 | Data Discovery | Profiling, quality checks, referential integrity |
| 2 | Data Cleaning | Standardisation, deduplication, type casting |


---
## Phase 0 — Setup
*Imports, configuration constants, and helper functions.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ── Paths ────────────────────────────────────────────────────────────────────
SOURCE_FILE = "/Users/ivanacaridad/Documents/GitHub/lumiere_analytics/data/raw/Lumiere_EU_Ecommerce.xlsx"

# ── Business constants ────────────────────────────────────────────────────────
SNAPSHOT_DATE    = pd.Timestamp("2026-01-01")   # RFM reference date
VALID_COUNTRIES  = ["France","Germany","Italy","Spain",
                    "Portugal","Netherlands","Belgium","Ireland"]
VALID_CATEGORIES = ["Apparel","Footwear","Accessories","Home & Living","Beauty"]
VALID_CHANNELS   = ["Web","Mobile App","Marketplace"]
VALID_SEGMENTS   = ["Consumer","Business","Premium"]
VALID_PAYMENT    = ["Credit Card","PayPal","Bank Transfer","Apple Pay","Klarna"]
COUNTRY_ALIASES  = {
    "Deutschland":"Germany","Allemagne":"Germany",
    "FR":"France","España":"Spain","Italia":"Italy",
    "Belgique":"Belgium","Pays-Bas":"Netherlands","Irlande":"Ireland",
}


print("✅  Setup complete")

✅  Setup complete


---
## Phase 1 — Data Discovery
*Load all tables, profile each one, run automated quality checks, and verify referential integrity.*


### 1.1  Load raw data

In [10]:
# Read all sheets at once into a dict keyed by sheet name
xl = pd.read_excel(SOURCE_FILE, sheet_name=None)

# Extract each sheet into a named DataFrame; .copy() 
orders    = xl["Orders"].copy()
products  = xl["Products"].copy()
customers = xl["Customers"].copy()
returns   = xl["Returns"].copy()
targets   = xl["Sales Targets"].copy()

# Print row and column counts per table to confirm the load
for name, df in [("Orders",orders),("Products",products),
                 ("Customers",customers),("Returns",returns),
                 ("Sales Targets",targets)]:
    print(f"  {name:<20} {len(df):>7,} rows  ×  {len(df.columns)} cols")

  Orders                82,000 rows  ×  12 cols
  Products                 177 rows  ×  8 cols
  Customers              5,200 rows  ×  7 cols
  Returns                6,150 rows  ×  3 cols
  Sales Targets            192 rows  ×  3 cols


### 1.2  Table previews

In [11]:
# Preview the first 3 rows of each table to verify column names and data types
print("── Orders ──"); display(orders.head(3))
print("── Products ──"); display(products.head(3))
print("── Customers ──"); display(customers.head(3))
print("── Returns ──"); display(returns.head(3))
print("── Sales Targets ──"); display(targets.head(3))

── Orders ──


,Order ID,Order Date,Ship Date,Customer ID,Product ID,Country,Channel,Quantity,Unit Price,Discount,Shipping Cost,Payment Method
0,EU-2024-000001,2024-01-01,2024-01-05,C14312,P1128,France,Web,1,148.86,0.20,4.90,Credit Card
1,EU-2024-000002,2024-01-01,2024-01-04,C10318,P1025,France,Mobile App,2,164.90,0.20,0.00,Credit Card
2,EU-2024-000003,2024-01-01,2024-01-03,C14095,P1045,Germany,Mobile App,1,235.53,0.20,0.00,Credit Card


── Products ──


,Product ID,Product Name,Category,Sub-Category,Brand,Unit Cost,List Price,Launch Date
0,P1001,Linen Shirt – Black,Apparel,Tops,Lumière Studio,46.07,124.06,2023-11-24
1,P1002,Linen Shirt – Burgundy,Apparel,Tops,North Coast,97.41,218.69,2022-02-02
2,P1003,Cotton T-Shirt – Navy,Apparel,Tops,Maison Lumière,41.40,90.30,2024-01-04


── Customers ──


,Customer ID,Customer Name,Segment,Country,City,Acquisition Date,Acquisition Channel
0,C10001,Hugo Maes,Business,Ireland,Galway,2023-09-16,Paid Search
1,C10002,Niamh Visser,Consumer,Germany,Berlin,2025-07-24,Organic Search
2,C10003,Marta Janssens,Premium,Spain,Barcelona,2024-06-26,Paid Search


── Returns ──


,Order ID,Return Date,Reason
0,EU-2025-071126,2025-12-01,Quality issue
1,EU-2024-002929,2024-04-06,Quality issue
2,EU-2025-038252,2025-06-16,Wrong size


── Sales Targets ──


,Country,Year-Month,Target Revenue
0,Germany,2024-01,"61,272.73"
1,Germany,2024-02,"63,636.36"
2,Germany,2024-03,"86,153.85"


### 1.3  Missing values & duplicates

In [12]:
def profile_nulls_dups(df, name):
    # Count missing values per column
    nulls = df.isnull().sum()
    # Count fully-duplicate rows
    dups  = df.duplicated().sum()
    # Express nulls as a percentage of total rows
    pct_null = (nulls / len(df) * 100).round(2)
    # Build a summary DataFrame showing only columns that have nulls
    summary  = pd.DataFrame({"nulls": nulls, "null_%": pct_null})
    summary  = summary[summary["nulls"] > 0]
    # Print section header with table name and duplicate count
    print(f"\n{'─'*45}")
    print(f"  {name}  |  rows: {len(df):,}  |  duplicates: {dups}")
    print(f"{'─'*45}")
    # Show null breakdown or confirm the table is clean
    if summary.empty:
        print("  No nulls found ✅")
    else:
        display(summary)

# Run the profiler on every table
for name, df in [("Orders",orders),("Products",products),
                 ("Customers",customers),("Returns",returns)]:
    profile_nulls_dups(df, name)


─────────────────────────────────────────────
  Orders  |  rows: 82,000  |  duplicates: 0
─────────────────────────────────────────────
  No nulls found ✅

─────────────────────────────────────────────
  Products  |  rows: 177  |  duplicates: 0
─────────────────────────────────────────────
  No nulls found ✅

─────────────────────────────────────────────
  Customers  |  rows: 5,200  |  duplicates: 0
─────────────────────────────────────────────
  No nulls found ✅

─────────────────────────────────────────────
  Returns  |  rows: 6,150  |  duplicates: 0
─────────────────────────────────────────────
  No nulls found ✅


### 1.4  Numerical column distributions

In [13]:
def detect_outliers_iqr(series):
    # First and third quartiles (25th and 75th percentiles)
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    # Interquartile range = spread of the middle 50% of values
    IQR = Q3 - Q1
    # Count values that fall outside the standard 1.5×IQR fences
    return int(((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum())

# Select the numeric columns most likely to contain business anomalies
num_cols = ["Unit Price","Discount","Quantity","Shipping Cost"]
# Get standard descriptive stats (count, mean, std, min, quartiles, max)
stats = orders[num_cols].describe().T
# Append the IQR outlier count as an extra diagnostic column
stats["iqr_outliers"] = [detect_outliers_iqr(orders[c]) for c in num_cols]
display(stats.round(3))

,count,mean,std,min,25%,50%,75%,max,iqr_outliers
Unit Price,"82,000.00",168.44,102.73,20.75,90.10,138.42,232.81,489.33,1598
Discount,"82,000.00",0.09,0.10,0.00,0.00,0.05,0.15,0.50,1864
Quantity,"82,000.00",1.61,0.95,1.00,1.00,1.00,2.00,5.00,4851
Shipping Cost,"82,000.00",2.76,2.68,0.00,0.00,4.90,4.90,6.50,0


### 1.5  Automated quality checks

In [ ]:
class DataQualityReport:
    def __init__(self):
        # List that accumulates all check results
        self.checks = []

    def add(self, table, check, result, detail=""):
        # result=True → PASS, False → FAIL; detail holds violation counts
        self.checks.append({
            "Table":  table,
            "Check":  check,
            "Status": "✅ PASS" if result else "❌ FAIL",
            "Detail": detail,
        })

    def dataframe(self):
        # Return all checks as a DataFrame for display and filtering
        return pd.DataFrame(self.checks)

# Instantiate the report collector
rpt = DataQualityReport()

# ── Orders ────────────────────────────────────────────────────────────────────
# No missing values across all columns
rpt.add("Orders","No null values",        orders.isnull().sum().sum()==0)
# Order ID must be unique — duplicates would double-count revenue
rpt.add("Orders","No duplicate Order IDs",not orders["Order ID"].duplicated().any())
# Discounts must stay within the business rule of 0–50%
rpt.add("Orders","Discount in [0, 0.5]",  orders["Discount"].between(0,0.5).all())
# All country values must belong to the approved EU market list
rpt.add("Orders","Valid countries",        orders["Country"].isin(VALID_COUNTRIES).all(),
        f"invalid: {orders[~orders['Country'].isin(VALID_COUNTRIES)]['Country'].unique()}")
# Channel must match one of the three known sales channels
rpt.add("Orders","Valid channels",         orders["Channel"].isin(VALID_CHANNELS).all())
# Ship date must not precede order date — negative lead times are impossible
rpt.add("Orders","Ship Date >= Order Date",
        (pd.to_datetime(orders["Ship Date"])>=pd.to_datetime(orders["Order Date"])).all())
# Unit prices must be positive — zero or negative prices indicate data errors
rpt.add("Orders","Unit Price > 0",         (orders["Unit Price"]>0).all())
# Quantity must fall within the allowed purchase range of 1–5 units
rpt.add("Orders","Quantity in [1,5]",      orders["Quantity"].between(1,5).all())
# Shipping cost can be zero (free shipping) but never negative
rpt.add("Orders","Shipping Cost >= 0",     (orders["Shipping Cost"]>=0).all())

# ── Products ──────────────────────────────────────────────────────────────────
rpt.add("Products","No null values",       products.isnull().sum().sum()==0)
# Product ID must be unique — duplicates would corrupt order enrichment joins
rpt.add("Products","No duplicate IDs",     not products["Product ID"].duplicated().any())
# Category must match the five defined merchandise divisions
rpt.add("Products","Valid categories",     products["Category"].isin(VALID_CATEGORIES).all())
# Unit cost must always be below list price to ensure positive gross margin
rpt.add("Products","Unit Cost < List Price",(products["Unit Cost"]<products["List Price"]).all())
# Compute gross margin ratio to check minimum profitability threshold
margin = (products["List Price"]-products["Unit Cost"])/products["List Price"]
rpt.add("Products","Margin > 10%",         (margin>0.10).all(),
        f"low-margin: {(margin<=0.10).sum()}")

# ── Customers ─────────────────────────────────────────────────────────────────
rpt.add("Customers","No null values",      customers.isnull().sum().sum()==0)
# Customer ID must be unique — duplicates would inflate customer counts
rpt.add("Customers","No duplicate IDs",    not customers["Customer ID"].duplicated().any())
# Segment must be one of the three defined tiers
rpt.add("Customers","Valid segments",      customers["Segment"].isin(VALID_SEGMENTS).all())
# Customer country must also belong to the eight valid EU markets
rpt.add("Customers","Valid countries",     customers["Country"].isin(VALID_COUNTRIES).all())

# ── Returns ───────────────────────────────────────────────────────────────────
# Each order can only be returned once — duplicates inflate the return rate
rpt.add("Returns","No duplicate records",  not returns["Order ID"].duplicated().any())
# Every return must reference an order that actually exists
rpt.add("Returns","Order IDs exist",       returns["Order ID"].isin(orders["Order ID"]).all())

# ── Referential integrity ─────────────────────────────────────────────────────
# Every order must have a matching customer record (no orphan orders)
rpt.add("Referential","Order → Customer FK",
        orders["Customer ID"].isin(customers["Customer ID"]).all())
# Every order must have a matching product record
rpt.add("Referential","Order → Product FK",
        orders["Product ID"].isin(products["Product ID"]).all())

# Display results as a colour-coded table (green = PASS, red = FAIL)
df_rpt = rpt.dataframe()

# Print overall pass rate as the final summary line
passed = df_rpt["Status"].str.contains("PASS").sum()
print(f"\nResult: {passed}/{len(df_rpt)} checks passed")

,Table,Check,Status,Detail
0,Orders,No null values,✅ PASS,
1,Orders,No duplicate Order IDs,✅ PASS,
2,Orders,"Discount in [0, 0.5]",✅ PASS,
3,Orders,Valid countries,✅ PASS,invalid: []
4,Orders,Valid channels,✅ PASS,
5,Orders,Ship Date >= Order Date,✅ PASS,
6,Orders,Unit Price > 0,✅ PASS,
7,Orders,"Quantity in [1,5]",✅ PASS,
8,Orders,Shipping Cost >= 0,✅ PASS,
9,Products,No null values,✅ PASS,



Result: 22/22 checks passed


---
## Phase 2 — Data Cleaning
*Apply standardisation, type casting, deduplication.*


In [15]:
def clean_orders(df):
    df = df.copy()   # avoid mutating the raw DataFrame
    # Parse date strings to datetime objects to enable date arithmetic
    df["Order Date"]    = pd.to_datetime(df["Order Date"])
    df["Ship Date"]     = pd.to_datetime(df["Ship Date"])
    # Normalize country names using the alias map, then title-case and strip whitespace
    df["Country"]       = df["Country"].replace(COUNTRY_ALIASES).str.strip().str.title()
    # Clamp discounts to the valid business range [0, 0.5]
    df["Discount"]      = df["Discount"].clip(0, 0.5)
    # Remove duplicate Order IDs, keeping the first occurrence
    before = len(df)
    df.drop_duplicates(subset=["Order ID"], keep="first", inplace=True)
    # Derive fulfillment speed as integer calendar days
    df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
    print(f"[Orders]    removed {before-len(df)} duplicates → {len(df):,} rows")
    return df

def clean_products(df):
    df = df.copy()
    # Parse launch date to datetime for tenure calculations
    df["Launch Date"]        = pd.to_datetime(df["Launch Date"])
    # Normalize category and brand text to prevent groupby mismatches
    df["Category"]           = df["Category"].str.strip().str.title()
    df["Brand"]              = df["Brand"].str.strip()
    # Pre-compute gross margin ratio for use in enrichment and KPI cells
    df["Product Margin Pct"] = (df["List Price"]-df["Unit Cost"])/df["List Price"]
    print(f"[Products]  {len(df):,} rows cleaned")
    return df

def clean_customers(df):
    df = df.copy()
    # Parse acquisition date to enable cohort and customer tenure calculations
    df["Acquisition Date"]    = pd.to_datetime(df["Acquisition Date"])
    # Normalize segment labels to title case
    df["Segment"]             = df["Segment"].str.strip().str.title()
    # Apply country alias map and normalize format
    df["Country"]             = df["Country"].replace(COUNTRY_ALIASES).str.strip().str.title()
    # Strip whitespace from the acquisition channel field
    df["Acquisition Channel"] = df["Acquisition Channel"].str.strip()
    print(f"[Customers] {len(df):,} rows cleaned")
    return df

def clean_returns(df):
    df = df.copy()
    # Parse return date to datetime
    df["Return Date"] = pd.to_datetime(df["Return Date"])
    # Normalize free-text reason field
    df["Reason"]      = df["Reason"].str.strip()
    # Keep only the first return record per order to remove duplicates
    before = len(df)
    df.drop_duplicates(subset=["Order ID"], keep="first", inplace=True)
    print(f"[Returns]   removed {before-len(df)} duplicates → {len(df):,} rows")
    return df

def clean_targets(df):
    df = df.copy()
    # Normalize country names to match the orders/customers format
    df["Country"] = df["Country"].str.strip().str.title()
    # Split "YYYY-MM" string into separate integer Year and Month columns for easier groupby
    df[["Year","Month"]] = df["Year-Month"].str.split("-", expand=True)
    df["Year"]  = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    print(f"[Targets]   {len(df):,} rows cleaned")
    return df

# ── Run cleaning pipeline ─────────────────────────────────────────────────────
# Apply all cleaning functions; results stored in *_c variables
orders_c    = clean_orders(orders)
products_c  = clean_products(products)
customers_c = clean_customers(customers)
returns_c   = clean_returns(returns)
targets_c   = clean_targets(targets)
print("\n✅  Cleaning complete")

[Orders]    removed 0 duplicates → 82,000 rows
[Products]  177 rows cleaned
[Customers] 5,200 rows cleaned
[Returns]   removed 0 duplicates → 6,150 rows
[Targets]   192 rows cleaned

✅  Cleaning complete


**Cleaning rules summary:**

| Field | Rule | Reason |
|---|---|---|
| Order/Ship Date | `pd.to_datetime` | Enables arithmetic |
| Country | Alias map + `.str.title()` | Consistent joins |
| Discount | `.clip(0, 0.5)` | Business constraint |
| Category / Brand | `.str.strip()` | Prevent groupby mismatches |
| Product Margin % | `(List − Cost) / List` | Early enrichment |
| Shipping Days | `Ship − Order` | Derived operational metric |

# Save cleaned datasets

In [16]:
# Save cleaned datasets
orders_c.to_parquet("../data/cleaned/orders_clean.parquet")
products_c.to_parquet("../data/cleaned/products_clean.parquet")
customers_c.to_parquet("../data/cleaned/customers_clean.parquet")
returns_c.to_parquet("../data/cleaned/returns_clean.parquet")
targets_c.to_parquet("../data/cleaned/targets_clean.parquet")